# Train & evaluate on Colab / Kaggle (thin wrapper)

Everything below calls the `captioner` package through `make`; no pipeline code lives here.

1. Runtime → Change runtime type → **T4 GPU** (or better).
2. Pick a config in the next cell and run the cells in order.
3. Copy `reports/<run>/metrics.json`, `predictions.csv` and figures back into the repo; never commit data, features or checkpoints.

| Config | Data | Encoder | Disk | Use |
|---|---|---|---|---|
| `configs/coco_clip.yaml` | MS-COCO 2014 (123k images) | CLIP ViT-B/32 | ≈ 50 GB | **recommended demo model** |
| `configs/coco_attention.yaml` | MS-COCO 2014 | ResNet-50 | ≈ 60 GB | COCO baseline |
| `configs/attention_neutral.yaml` | Flickr8k | ResNet-50 | ≈ 5 GB | course benchmark, neutral captions |
| `configs/attention.yaml` / `merge_baseline.yaml` | Flickr8k | ResNet-50 | ≈ 5 GB | blueprint experiments E1–E4 |

Colab's free disk fits COCO only if nothing else is stored; on Kaggle use a Flickr8k config or attach the COCO 2014 dataset and set `dataset.sources[0].image_root` to its absolute path.

In [ ]:
REPO = 'https://github.com/Kislay01/DeepLearning_CP.git'
CONFIG = 'configs/coco_clip.yaml'

In [ ]:
!git clone --depth 1 $REPO captioner_repo
%cd captioner_repo
!pip install -q uv
!java -version  # METEOR + PTB tokenizer need Java (preinstalled on Colab)
!df -h /content | tail -1

In [ ]:
!uv sync --frozen

## Pipeline
`make data` downloads the dataset (checksums verified or recorded on first use) and builds the manifest + train-only vocabulary; `make features` caches encoder features to HDF5; `make train` trains with early stopping on val BLEU-4 and then calibrates the out-of-domain detector and confidence threshold; `make eval` scores the test split with beam 1/3/5 and writes fairness and reliability checks.

In [ ]:
!make data CONFIG=$CONFIG
if 'coco' not in CONFIG and 'combined' not in CONFIG:
    !uv run python scripts/verify_data.py --config $CONFIG --skip-md5

In [ ]:
!make features CONFIG=$CONFIG

In [ ]:
!make train CONFIG=$CONFIG

In [ ]:
!make eval CONFIG=$CONFIG

In [ ]:
!make export CONFIG=$CONFIG
!ls -la reports/*/ artifacts/release/*/
!cat artifacts/release/*/release.yaml

## Optional
- 3 seeds: `!uv run captioner train -c $CONFIG --set seed=1` (and `seed=2`).
- Ablations and every other experiment: see `reports/README.md`.
- TensorBoard: `%load_ext tensorboard` then `%tensorboard --logdir artifacts/runs`.
- Publish: upload `artifacts/release/<version>/*` to a GitHub Release, copy `release.yaml` into `configs/release.yaml` with `base_url` set, push — the app and Docker image then download and verify it.